# S8 - Procesamiento en streaming con Spark: ventanas, watermarking y semantica de entrega

**Actividad:** construir un pipeline de Spark Structured Streaming sobre `atmos-eventos` (el mismo topic de Kafka que S7 ya deja publicando telemetria de sensores), aplicando agregaciones por ventana de tiempo, watermarking para datos tardios, checkpointing para tolerancia a fallos, deduplicacion acotada, y comparando semantica de entrega y throughput/latencia bajo distintos parametros.


## 3.1 Reanudar el entorno `lambda26` y verificar el punto de partida

**Producto del paso:** Kafka, Kafka UI y el simulador de sensores de S7 (`uso-atmos`) corriendo, con eventos reales llegando a `atmos-eventos`; y `pyspark` unido a la red de Kafka, algo que hasta hoy nunca necesito.

Este notebook **no genera sus propios datos**: consume el mismo topic que S7 ya construyo. Antes de continuar, en una terminal (fuera de este notebook):

```bash
cd kafka && docker compose up -d
cd ../uso-atmos && docker compose up -d
docker exec -d lambda26-uso-atmos python /app/producer_sensores.py
```

Deja el productor corriendo durante toda la sesion: varias celdas de este notebook necesitan eventos **llegando en vivo**, no solo los que ya estan en el topic.

`pyspark/compose.yml` corre desde S1, y nunca tuvo que hablar con Kafka: hasta S7, Kafka lo usaban otros contenedores (`uso-atmos`, `ec-eventos-py`), nunca Spark. Hoy cambia: `spark.read.format("kafka")` (mas abajo) corre **dentro** del contenedor `pyspark`, y ese contenedor todavia no esta en la misma red que `lambda26-kafka` — sin unirlo, `kafka:9092` no se resuelve. En vez de tocar `pyspark/compose.yml` (que sirve para todo el curso, no solo para streaming), crea un segundo archivo que solo agrega esa red:

**`pyspark/compose.kafka.yml`:**

```yaml
services:
  pyspark:
    networks:
      - default
      - lambda26-kafka-net

networks:
  lambda26-kafka-net:
    external: true
    name: lambda26-kafka-net
```

Subelo combinando los dos archivos — el segundo solo agrega la red, no reemplaza nada del primero:

```bash
cd ../pyspark
docker compose -f compose.yml -f compose.kafka.yml up -d --build
```

**Error frecuente**: la celda de abajo falla con `kafka.errors.NoBrokersAvailable` o, en el log del driver, `java.net.UnknownHostException: kafka`. `pyspark` esta corriendo, pero sin `compose.kafka.yml` — subelo de nuevo con los dos `-f` del comando de arriba.
</cell>


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion8-streaming-estructurado")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1c263b0f-0b10-4b0c-a722-959fe77d3f8b;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central


	found org.slf4j#slf4j-api;2.0.17 in central


...

	[SUCCESSFUL ] at.yawk.lz4#lz4-java;1.11.0!lz4-java.jar (246ms)
downloading https://repo1.maven.org/maven2/org/xerial/snappy/snappy-java/1.1.10.8/snappy-java-1.1.10.8.jar ...


	[SUCCESSFUL ] org.xerial.snappy#snappy-java;1.1.10.8!snappy-java.jar(bundle) (363ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/2.0.17/slf4j-api-2.0.17.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;2.0.17!slf4j-api.jar (138ms)


downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-api/3.5.0/hadoop-client-api-3.5.0.jar ...


	[SUCCESSFUL ] org.apache.hadoop#hadoop-client-api;3.5.0!hadoop-client-api.jar (1302ms)
:: resolution report :: resolve 11956ms :: artifacts dl 5865ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	----------------------------------------------------------------

26/10/03 14:00:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


`spark.jars.packages` agrega el conector `spark-sql-kafka-0-10`, que Spark no trae por defecto (a diferencia de la lectura de CSV o Parquet de S2-S4). La primera vez que se ejecuta esta celda, Spark descarga el conector y sus dependencias desde Maven Central — puede tardar. `spark.sql.shuffle.partitions` se baja a 3 (el numero de particiones del topic, S7): con el valor por defecto (200), cada micro-lote crearia 200 tareas para casi ningun dato.

Verifica que el topic tiene mensajes, con una lectura **batch** (no streaming) — el mismo tipo de lectura que ya usaste con CSV o Parquet, aplicada a Kafka:


In [2]:
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en el topic:", lote.count())
lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset").orderBy("offset", ascending=False).show(3, truncate=False)


mensajes en el topic: 102


+-----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|key              |value                                                                                                                                                                       |partition|offset|
+-----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|esp32-invernadero|{"tipoEvento": "sensor.lectura", "sensorId": "esp32-invernadero", "temperatura": 37.2, "humedad": 59.8, "presion": 995.3, "origen": "uso-atmos", "timestamp": 1791036066053}|2        |2513  |
|esp32-patio      |{"tipoEvento": "sensor.lectura", "sensorId": "esp32-patio", "temperatura": 19.6, "humedad": 63.1, "presion": 1005.9, "origen": "uso-atmos", "

Resultado esperado: un numero mayor que cero, y en las ultimas filas, JSON con `sensorId`, `temperatura`, `humedad`, `presion` (el contrato de S7). Una lectura **batch** de Kafka lee lo que hay **hoy** y termina; el resto del notebook usa lectura **streaming**, que no termina — sigue escuchando.


## 3.2 Leer el topic en modo streaming

**Producto del paso:** un DataFrame que **no se puede materializar con `.show()` ni `.count()`** — la primera diferencia real entre batch y streaming.


In [3]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)
crudo.printSchema()


es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



`readStream` en vez de `read` es la unica diferencia de sintaxis, y cambia todo lo que viene despues: `crudo` es un **plan**, no una tabla en memoria. `crudo.show()` fallaria con `AnalysisException`, porque mostrar implica materializar de una vez, y un stream no tiene un "final" que mostrar. Cada mensaje de Kafka llega con las mismas columnas, sin importar el topic: `key`, `value` (ambos en binario), `topic`, `partition`, `offset`, `timestamp`. El contenido real del evento esta **adentro** de `value`, sin parsear todavia — eso es 3.3.

`startingOffsets: "latest"` (no `"earliest"`) es a proposito: en un stream que corre indefinidamente, no tiene sentido reprocesar todo el historial cada vez que se reinicia una consulta de prueba — se quiere ver lo que llega **de ahora en adelante**. La diferencia importa, y se vuelve a ver en 3.9.


## 3.3 Parsear el JSON con un esquema explicito

**Producto del paso:** las columnas reales del evento (`sensorId`, `temperatura`...), ya tipadas, listas para agregar.


In [4]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)
eventos.printSchema()


root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



`value` llega en binario; `CAST(value AS STRING)` lo vuelve texto, y `from_json` lo convierte a columnas segun `esquema_evento` — el mismo contrato de `sensor.lectura` que S7 documento. Un esquema **explicito** (no inferido) es obligatorio en streaming: Spark no puede "mirar" todo el stream para adivinar tipos, porque el stream no tiene fin. `ts` convierte el `timestamp` (milisegundos *epoch*, un `Long`) a un tipo `timestamp` real: **es la columna que 3.5 en adelante usa como reloj del evento** — el momento en que el sensor midio, no el momento en que Spark lo procesa.


## 3.4 Primera consulta: ver los eventos llegar

**Producto del paso:** la primera consulta de streaming corriendo, con salida por consola.


In [5]:
consulta = (
    eventos.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(20)
consulta.stop()


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+--------+-----------+-------+-------+------+---------+---+
|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|ts |
+----------+--------+-----------+-------+-------+------+---------+---+
+----------+--------+-----------+-------+-------+------+---------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|36.1       |33.4   |1000.6 |uso-atmos|1791036069058|2026-10-03 14:01:09.058|
|sensor.lectura|esp32-patio      |19.4       |62.4   |1005.9 |uso-atmos|1791036069055|2026-10-03 14:01:09.055|
|sensor.lectura|esp32-invernadero|37.0       |59.9   |995.3  |uso-atmos|1791036069057|2026-10-03 14:01:09.057|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|36.0       |32.8   |1000.4 |uso-atmos|1791036072070|2026-10-03 14:01:12.07 |
|sensor.lectura|esp32-patio      |19.3       |61.9   |1006.1 |uso-atmos|1791036072059|2026-10-03 14:01:12.059|
|sensor.lectura|esp32-invernadero|37.3       |58.9   |995.0  |uso-atmos|1791036072068|2026-10-03 14:01:12.068|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|35.7       |33.8   |1000.8 |uso-atmos|1791036075075|2026-10-03 14:01:15.075|
|sensor.lectura|esp32-laboratorio|35.9       |34.2   |1000.5 |uso-atmos|1791036078079|2026-10-03 14:01:18.079|
|sensor.lectura|esp32-patio      |19.4       |61.0   |1005.9 |uso-atmos|1791036075071|2026-10-03 14:01:15.071|
|sensor.lectura|esp32-invernadero|37.1       |59.3   |995.2  |uso-atmos|1791036075074|2026-10-03 14:01:15.074|
|sensor.lectura|esp32-patio      |19.7       |61.8   |1005.4 |uso-atmos|1791036078076|2026-10-03 14:01:18.076|
|sensor.lectura

-------------------------------------------
Batch: 4
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|36.1       |33.3   |1000.1 |uso-atmos|1791036081084|2026-10-03 14:01:21.084|
|sensor.lectura|esp32-laboratorio|36.1       |32.9   |999.9  |uso-atmos|1791036084090|2026-10-03 14:01:24.09 |
|sensor.lectura|esp32-patio      |19.6       |62.5   |1005.8 |uso-atmos|1791036081081|2026-10-03 14:01:21.081|
|sensor.lectura|esp32-invernadero|37.1       |60.3   |995.0  |uso-atmos|1791036081083|2026-10-03 14:01:21.083|
|sensor.lectura|esp32-patio      |19.6       |62.5   |1006.1 |uso-atmos|1791036084086|2026-10-03 14:01:24.086|
|sensor.lectura

`writeStream` (no `write`) inicia una **consulta continua**: `.start()` la lanza en segundo plano y devuelve de inmediato un objeto `StreamingQuery`; el proceso sigue corriendo aunque la celda ya haya terminado. `awaitTermination(20)` bloquea esta celda 20 segundos para que se alcance a ver algo, y `.stop()` la detiene a mano — en un sistema real, una consulta de streaming no se detiene sola, corre para siempre.

`outputMode("append")` dice que solo se muestran filas **nuevas**, nunca modificadas — es el unico modo valido para eventos sin agregar, como estos. `trigger(processingTime="5 seconds")` fija cada cuanto se arma un micro-lote: Structured Streaming no procesa evento por evento, sino en **micro-lotes** cada 5 segundos. Resultado esperado: varias tandas de filas, una por cada micro-lote, con los mismos campos que en 3.1.


## 3.5 Agregar por ventana de tiempo, sin limite (el problema que resuelve 3.6)

**Producto del paso:** evidencia de que, sin watermark, el estado de una agregacion por ventana **crece para siempre**.


In [6]:
from pyspark.sql.functions import window, avg, count

agregado_sin_limite = (
    eventos.groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_sin_limite.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(25)
consulta.stop()


-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---------+---+
|window|sensorId|temp_prom|n  |
+------+--------+---------+---+
+------+--------+---------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-invernadero|37.0     |1  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-patio      |19.9     |1  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-laboratorio|36.3     |1  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-invernadero|37.0     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-laboratorio|36.55    |2  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-patio      |19.9     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-patio      |19.9     |2  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-invernadero|36.7     |2  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-laboratorio|36.3     |1  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-invernadero|37.0     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-laboratorio|36.5     |4  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-patio      |19.9     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-patio      |19.9     |4  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-invernadero|36.6     |4  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-laboratorio|36.3     |1  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-invernadero|37.0     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-laboratorio|36.5     |4  |
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}|esp32-invernadero|36.6     |1  |
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}|esp32-patio      |19.8     |1  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-patio      |19.9     |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-patio      |19.9     |4  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-invernadero|36.6     |4  |
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}|esp32-laboratorio|36.8     |1  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-labora

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+-----------------+------------------+---+
|window                                    |sensorId         |temp_prom         |n  |
+------------------------------------------+-----------------+------------------+---+
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}|esp32-invernadero|36.800000000000004|3  |
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}|esp32-patio      |19.766666666666666|3  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-patio      |19.9              |1  |
|{2026-10-03 14:01:20, 2026-10-03 14:01:30}|esp32-invernadero|37.0              |1  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-laboratorio|36.5              |4  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-patio      |19.9              |4  |
|{2026-10-03 14:01:30, 2026-10-03 14:01:40}|esp32-invernadero|36.6              |4  |
|{2026-10-03 14:01:40, 2026-10-03 14:01:50}

`window(col("ts"), "10 seconds")` agrupa los eventos en bloques de 10 segundos de tiempo de **evento** (`ts`), no de tiempo de procesamiento — dos sensores que midieron en el mismo instante caen en la misma ventana, sin importar cuando Spark los proceso. `outputMode("complete")` es el unico modo que puede acompanar a una agregacion **sin** watermark: en cada micro-lote, Spark reimprime **todas** las ventanas vistas hasta ahora, porque no tiene ninguna senal de que una ventana vieja ya no va a recibir mas datos.

Observa que, de un micro-lote al siguiente, la tabla completa **crece**: las ventanas de hace 20 segundos siguen apareciendo, iguales, junto a las nuevas. Ese es el problema: Spark tiene que guardar el estado de **cada ventana que existio jamas**, indefinidamente — en un sistema real, que corre por dias, esto agota la memoria. El watermarking (3.6) es la respuesta.


## 3.6 Watermarking: acotar cuanto se espera por datos tardios

**Producto del paso:** la misma agregacion, pero con las ventanas viejas **cerradas y liberadas** de la memoria.


In [7]:
agregado_con_watermark = (
    eventos
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_con_watermark.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-watermark")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(25)
consulta.stop()


-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---------+---+
|window|sensorId|temp_prom|n  |
+------+--------+---------+---+
+------+--------+---------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-invernadero|36.9     |1  |
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-patio      |20.0     |1  |
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-laboratorio|36.7     |1  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-invernadero|37.0     |2  |
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-patio      |19.9     |2  |
|{2026-10-03 14:01:50, 2026-10-03 14:02:00}|esp32-laboratorio|36.8     |2  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-----------------+------------------+---+
|window                                    |sensorId         |temp_prom         |n  |
+------------------------------------------+-----------------+------------------+---+
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-invernadero|37.150000000000006|2  |
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-laboratorio|36.5              |2  |
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-patio      |19.55             |2  |
+------------------------------------------+-----------------+------------------+---+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-invernadero|37.125   |4  |
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-laboratorio|36.55    |4  |
|{2026-10-03 14:02:00, 2026-10-03 14:02:10}|esp32-patio      |19.4     |4  |
+------------------------------------------+-----------------+---------+---+



-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:02:10, 2026-10-03 14:02:20}|esp32-laboratorio|36.6     |1  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:20}|esp32-patio      |19.2     |1  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:20}|esp32-invernadero|37.4     |1  |
+------------------------------------------+-----------------+---------+---+



`withWatermark("ts", "10 seconds")` es la regla que le faltaba a 3.5: le dice a Spark "una vez que veas un evento con `ts` = X, ya no esperes datos con `ts` anterior a X menos 10 segundos — cierra esas ventanas y olvidalas". El **watermark** en un momento dado es, entonces, el evento mas reciente visto menos ese margen. Con el watermark, `outputMode("update")` ya es valido: en cada micro-lote solo se imprimen las ventanas que **cambiaron**, no todas — y las ventanas cerradas dejan de aparecer y dejan de ocupar memoria.

`checkpointLocation` aparece por primera vez: es la carpeta donde Spark guarda el estado de la consulta (que ventanas existen, hasta que offset de Kafka se leyo). Sin ella, esta consulta con estado (una agregacion) ni siquiera arrancaria. Se profundiza en 3.8.

Compara la salida con la de 3.5: aca la tabla **no** repite ventanas viejas en cada lote — cada micro-lote solo trae lo nuevo o lo que cambio.


## 3.7 Ventana deslizante: una lectura que se actualiza mas seguido que se cierra

**Producto del paso:** ventanas que se **solapan**, para suavizar una metrica sin esperar a que cada bloque termine.


In [8]:
agregado_deslizante = (
    eventos
    .withWatermark("ts", "15 seconds")
    .groupBy(window(col("ts"), "15 seconds", "5 seconds"), col("sensorId"))
    .agg(avg("temperatura").alias("temp_prom"), count("*").alias("n"))
)

consulta = (
    agregado_deslizante.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-deslizante")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(25)
consulta.stop()


-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---------+---+
|window|sensorId|temp_prom|n  |
+------+--------+---------+---+
+------+--------+---------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-----------------+---------+---+
|window                                    |sensorId         |temp_prom|n  |
+------------------------------------------+-----------------+---------+---+
|{2026-10-03 14:02:10, 2026-10-03 14:02:25}|esp32-laboratorio|36.4     |1  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-patio      |19.1     |1  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:25}|esp32-patio      |19.1     |1  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-invernadero|37.8     |1  |
|{2026-10-03 14:02:05, 2026-10-03 14:02:20}|esp32-invernadero|37.8     |1  |
|{2026-10-03 14:02:05, 2026-10-03 14:02:20}|esp32-patio      |19.1     |1  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-laboratorio|36.4     |1  |
|{2026-10-03 14:02:05, 2026-10-03 14:02:20}|esp32-laboratorio|36.4     |1  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:25}|esp32-invern

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-----------------+------------------+---+
|window                                    |sensorId         |temp_prom         |n  |
+------------------------------------------+-----------------+------------------+---+
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-laboratorio|36.05             |2  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:25}|esp32-laboratorio|36.166666666666664|3  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-patio      |19.299999999999997|2  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-patio      |19.23333333333333 |3  |
|{2026-10-03 14:02:10, 2026-10-03 14:02:25}|esp32-patio      |19.23333333333333 |3  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-invernadero|37.9              |2  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-invernadero|37.86666666666667 |3  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-----------------+-----------------+---+
|window                                    |sensorId         |temp_prom        |n  |
+------------------------------------------+-----------------+-----------------+---+
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-laboratorio|35.7             |1  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-laboratorio|35.93333333333333|3  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-patio      |19.6             |1  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-patio      |19.4             |3  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-patio      |19.325           |4  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-invernadero|38.0             |1  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-invernadero|37.93333333333333|3  |
|{2026-10-03 14:02:15, 2026-10-03 14:02:30}|esp32-inv

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+-----------------+------------------+---+
|window                                    |sensorId         |temp_prom         |n  |
+------------------------------------------+-----------------+------------------+---+
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-laboratorio|35.6              |3  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-laboratorio|35.779999999999994|5  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-patio      |19.73333333333333 |3  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-patio      |19.559999999999995|5  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-invernadero|38.06666666666667 |3  |
|{2026-10-03 14:02:20, 2026-10-03 14:02:35}|esp32-invernadero|38.0              |5  |
|{2026-10-03 14:02:30, 2026-10-03 14:02:45}|esp32-laboratorio|35.55             |2  |
|{2026-10-03 14:02:30, 2026-10-03 14:02:45}

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+-----------------+------------------+---+
|window                                    |sensorId         |temp_prom         |n  |
+------------------------------------------+-----------------+------------------+---+
|{2026-10-03 14:02:35, 2026-10-03 14:02:50}|esp32-laboratorio|35.55             |2  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-laboratorio|35.58             |5  |
|{2026-10-03 14:02:35, 2026-10-03 14:02:50}|esp32-patio      |19.700000000000003|2  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-patio      |19.72             |5  |
|{2026-10-03 14:02:25, 2026-10-03 14:02:40}|esp32-invernadero|37.98             |5  |
|{2026-10-03 14:02:30, 2026-10-03 14:02:45}|esp32-laboratorio|35.55             |4  |
|{2026-10-03 14:02:30, 2026-10-03 14:02:45}|esp32-patio      |19.75             |4  |
|{2026-10-03 14:02:35, 2026-10-03 14:02:50}

`window(col("ts"), "15 seconds", "5 seconds")` tiene un tercer argumento que 3.6 no tenia: el **paso** (5 segundos) es menor que el **ancho** (15 segundos) de la ventana. El resultado es una ventana **deslizante**: cada evento cae en **varias** ventanas de 15 segundos a la vez (una que empieza ahora, otra que empezo hace 5 segundos, otra hace 10). En 3.6, cada evento caia en **una sola** ventana (ventana **fija** o *tumbling*: el paso es igual al ancho).

Observa en la salida que, para un mismo `sensorId`, aparecen varias filas con rangos de `window` que se superponen — y el mismo evento cuenta para varias de ellas. Sirve para suavizar una lectura (un promedio que no salta de golpe cada 15 segundos), a costa de guardar mas estado: cada evento se cuenta varias veces en la memoria de la consulta, no una.


## 3.8 Checkpointing: que guarda, y que pasa al reiniciar

**Producto del paso:** evidencia de que el *checkpoint* no es una carpeta vacia — ahi vive el progreso real de la consulta.


In [9]:
import os

for carpeta in sorted(os.listdir("/tmp/s08-chk-watermark")):
    print(carpeta)


.metadata.crc
commits
metadata
offsets
sources
state


In [10]:
archivos_offsets = sorted(os.listdir("/tmp/s08-chk-watermark/offsets"))
ultimo_offset = archivos_offsets[-1]
with open(f"/tmp/s08-chk-watermark/offsets/{ultimo_offset}") as f:
    print(f.read())


v1
{"batchWatermarkMs":1791036119175,"batchTimestampMs":1791036135010,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.function.protobufExtensions.enabled":"false","spark.sql.shuffle.partitions":"3","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.optimizer.pruneFiltersCanPrune

`offsets/` guarda, por cada micro-lote, hasta que *offset* de cada particion de Kafka se leyo. `commits/` confirma cuales de esos micro-lotes terminaron de escribirse por completo. `state/` guarda el contenido de cada ventana abierta — es lo que 3.5 mostro creciendo sin limite.

La consecuencia practica: si una consulta se cae y se reinicia **apuntando al mismo `checkpointLocation`**, Spark lee `offsets/` y retoma exactamente donde se quedo — no vuelve a leer el topic desde el principio, y no pierde el estado de las ventanas que ya tenia abiertas. Es la misma idea del *offset* de un `consumer group` de Kafka (S6, S7), pero aplicada tambien al estado de la agregacion, no solo a la posicion de lectura.


## 3.9 Un dato tardio, y el watermark descartandolo

**Producto del paso:** un evento con marca de tiempo vieja, publicado **despues** de que el watermark ya avanzo mas alla de su ventana — y la prueba de que Spark lo descarta sin avisar.

**Advertencia sobre el "arranque en frio":** si publicas el evento tardio como el **primer** mensaje que la consulta ve, no se descarta — el watermark todavia no tiene ningun evento "reciente" contra el cual compararlo. Por eso, primero se deja correr la consulta con datos en vivo un rato, y **recien despues** se publica el evento tardio.


In [11]:
import threading
import time
import json
from pyspark.sql.functions import to_json, struct

def publicar_evento_tardio(segundos_de_espera, segundos_en_el_pasado):
    time.sleep(segundos_de_espera)
    ahora_ms = int(time.time() * 1000)
    evento_tardio = [(
        "sensor.lectura", "esp32-tardio", 99.9, 50.0, 1000.0,
        "prueba-watermark", ahora_ms - segundos_en_el_pasado * 1000
    )]
    columnas = ["tipoEvento", "sensorId", "temperatura", "humedad", "presion", "origen", "timestamp"]
    df_tardio = spark.createDataFrame(evento_tardio, columnas)
    salida = df_tardio.select(
        df_tardio.sensorId.cast("string").alias("key"),
        to_json(struct(*columnas)).alias("value"),
    )
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "atmos-eventos")
        .save())
    print(f">>> publicado esp32-tardio, {segundos_en_el_pasado}s en el pasado")

hilo = threading.Thread(target=publicar_evento_tardio, args=(15, 45), daemon=True)
hilo.start()

agregado = (
    eventos
    .withWatermark("ts", "5 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"))
)

consulta = (
    agregado.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-tardio")
    .trigger(processingTime="4 seconds")
    .start()
)
consulta.awaitTermination(45)
consulta.stop()
print("Buscando 'esp32-tardio' en la salida de arriba: si no aparece ninguna fila, el watermark lo descarto.")


-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---+
|window|sensorId|n  |
+------+--------+---+
+------+--------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-laboratorio|1  |
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-patio      |1  |
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-invernadero|1  |
+------------------------------------------+-----------------+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-laboratorio|2  |
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-patio      |2  |
|{2026-10-03 14:02:40, 2026-10-03 14:02:50}|esp32-invernadero|2  |
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-patio      |1  |
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-invernadero|1  |
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-laboratorio|1  |
+------------------------------------------+-----------------+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-patio      |2  |
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-invernadero|2  |
|{2026-10-03 14:02:50, 2026-10-03 14:03:00}|esp32-laboratorio|2  |
+------------------------------------------+-----------------+---+



>>> publicado esp32-tardio, 45s en el pasado


...

-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:03:00, 2026-10-03 14:03:10}|esp32-invernadero|4  |
|{2026-10-03 14:03:00, 2026-10-03 14:03:10}|esp32-laboratorio|4  |
|{2026-10-03 14:03:00, 2026-10-03 14:03:10}|esp32-patio      |4  |
+------------------------------------------+-----------------+---+



-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-patio      |2  |
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-laboratorio|2  |
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-invernadero|2  |
+------------------------------------------+-----------------+---+



-------------------------------------------
Batch: 9
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-patio      |3  |
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-laboratorio|3  |
|{2026-10-03 14:03:10, 2026-10-03 14:03:20}|esp32-invernadero|3  |
+------------------------------------------+-----------------+---+



-------------------------------------------
Batch: 10
-------------------------------------------
+------------------------------------------+-----------------+---+
|window                                    |sensorId         |n  |
+------------------------------------------+-----------------+---+
|{2026-10-03 14:03:20, 2026-10-03 14:03:30}|esp32-laboratorio|1  |
|{2026-10-03 14:03:20, 2026-10-03 14:03:30}|esp32-patio      |1  |
|{2026-10-03 14:03:20, 2026-10-03 14:03:30}|esp32-invernadero|1  |
+------------------------------------------+-----------------+---+



Buscando 'esp32-tardio' en la salida de arriba: si no aparece ninguna fila, el watermark lo descarto.


El hilo publica un evento `esp32-tardio` cuyo `timestamp` corresponde a **45 segundos antes** del momento de publicacion, pero recien 15 segundos despues de arrancar la consulta — tiempo suficiente para que el watermark ya haya avanzado con los eventos en vivo del productor de S7. Con un watermark de 5 segundos, cualquier ventana que termine mas de 5 segundos antes del evento mas reciente visto queda cerrada, y un evento que llega despues para esa ventana se descarta **en silencio**: no hay excepcion, no hay log de error, la fila simplemente nunca aparece en la salida.

Esto se verifico con datos reales antes de escribir este notebook: el mismo experimento, publicando el evento tardio **antes** de que el watermark avanzara (como primer mensaje de la consulta), si lo dejaba pasar — es el caso de "arranque en frio" de la advertencia de arriba. La leccion no es "Spark falla a veces", es que el watermark necesita **datos frescos previos** para tener algo contra qué comparar.


## 3.10 Deduplicacion acotada por watermark

**Producto del paso:** el mismo evento, publicado dos veces por error, contado **una sola vez**.


In [12]:
deduplicado = (
    eventos
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "timestamp"])
)

consulta = (
    deduplicado.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-dedup")
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(6)
columnas = ["tipoEvento", "sensorId", "temperatura", "humedad", "presion", "origen", "timestamp"]
evento_duplicado = spark.createDataFrame(
    [("sensor.lectura", "esp32-duplicado", 55.0, 55.0, 1000.0, "prueba-dedup", int(time.time() * 1000))],
    columnas,
)
salida = evento_duplicado.select(
    evento_duplicado.sensorId.cast("string").alias("key"),
    to_json(struct(*columnas)).alias("value"),
)
for intento in range(2):
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "atmos-eventos")
        .save())
print(">>> 'esp32-duplicado' publicado DOS veces")

consulta.awaitTermination(20)
consulta.stop()
print("Busca 'esp32-duplicado' arriba: debe aparecer una sola vez, aunque se publico dos veces.")


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-patio      |19.9       |66.4   |1005.2 |uso-atmos|1791036207369|2026-10-03 14:03:27.369|
|sensor.lectura|esp32-invernadero|37.5       |57.7   |995.0  |uso-atmos|1791036207371|2026-10-03 14:03:27.371|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|35.3       |31.7   |999.9  |uso-atmos|1791036207372|2026-10-03 14:03:27.372|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-invernadero|37.5       |57.4   |995.1  |uso-atmos|1791036210381|2026-10-03 14:03:30.381|
|sensor.lectura|esp32-laboratorio|35.5       |31.3   |999.5  |uso-atmos|1791036210383|2026-10-03 14:03:30.383|
|sensor.lectura|esp32-patio      |20.1       |65.6   |1005.3 |uso-atmos|1791036210373|2026-10-03 14:03:30.373|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+

>>> 'esp32-duplicado' publicado DOS veces


-------------------------------------------
Batch: 3
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+------------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen      |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+------------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|35.5       |31.0   |999.6  |uso-atmos   |1791036213391|2026-10-03 14:03:33.391|
|sensor.lectura|esp32-patio      |20.3       |64.8   |1004.9 |uso-atmos   |1791036213384|2026-10-03 14:03:33.384|
|sensor.lectura|esp32-duplicado  |55.0       |55.0   |1000.0 |prueba-dedup|1791036213332|2026-10-03 14:03:33.332|
|sensor.lectura|esp32-invernadero|37.7       |57.1   |995.1  |uso-atmos   |1791036213388|2026-10-03 14:03:33.388|
+--------------+-----------------+-----------+-------+-------+------------+-------------+----------------

-------------------------------------------
Batch: 4
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-invernadero|37.9       |56.8   |995.4  |uso-atmos|1791036216403|2026-10-03 14:03:36.403|
|sensor.lectura|esp32-laboratorio|35.8       |31.0   |999.4  |uso-atmos|1791036216404|2026-10-03 14:03:36.404|
|sensor.lectura|esp32-patio      |20.1       |63.9   |1005.1 |uso-atmos|1791036216395|2026-10-03 14:03:36.395|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 5
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-patio      |19.9       |63.2   |1005.5 |uso-atmos|1791036219406|2026-10-03 14:03:39.406|
|sensor.lectura|esp32-laboratorio|36.0       |30.1   |999.9  |uso-atmos|1791036219415|2026-10-03 14:03:39.415|
|sensor.lectura|esp32-invernadero|38.0       |57.4   |995.4  |uso-atmos|1791036219414|2026-10-03 14:03:39.414|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 6
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|36.2       |30.1   |999.5  |uso-atmos|1791036222430|2026-10-03 14:03:42.43 |
|sensor.lectura|esp32-invernadero|38.3       |58.0   |995.5  |uso-atmos|1791036222421|2026-10-03 14:03:42.421|
|sensor.lectura|esp32-patio      |19.8       |62.2   |1005.4 |uso-atmos|1791036222418|2026-10-03 14:03:42.418|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 7
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-laboratorio|36.1       |30.4   |1000.0 |uso-atmos|1791036225443|2026-10-03 14:03:45.443|
|sensor.lectura|esp32-patio      |20.0       |61.4   |1005.0 |uso-atmos|1791036225432|2026-10-03 14:03:45.432|
|sensor.lectura|esp32-invernadero|38.3       |57.6   |995.4  |uso-atmos|1791036225441|2026-10-03 14:03:45.441|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



-------------------------------------------
Batch: 8
-------------------------------------------
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |ts                     |
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+
|sensor.lectura|esp32-patio      |19.8       |61.3   |1005.3 |uso-atmos|1791036228445|2026-10-03 14:03:48.445|
|sensor.lectura|esp32-invernadero|38.2       |57.1   |995.0  |uso-atmos|1791036228447|2026-10-03 14:03:48.447|
|sensor.lectura|esp32-laboratorio|35.9       |30.5   |999.7  |uso-atmos|1791036228448|2026-10-03 14:03:48.448|
+--------------+-----------------+-----------+-------+-------+---------+-------------+-----------------------+



Busca 'esp32-duplicado' arriba: debe aparecer una sola vez, aunque se publico dos veces.


`dropDuplicates(["sensorId", "timestamp"])` descarta cualquier evento cuya combinacion de esas dos columnas ya se vio antes. Sin `withWatermark`, esta operacion tendria que recordar **todos** los eventos vistos jamas, para siempre — el mismo problema sin limite de 3.5, pero sobre los propios datos en vez de sobre ventanas. El watermark acota cuanto se recuerda: pasado el margen (30 segundos), un identificador viejo se olvida, y un duplicado *muy* tardio ya no se detectaria — es el mismo balance de siempre entre memoria y exactitud.

Esto responde directamente al porque de **semantica de entrega**: Kafka entrega cada mensaje **al menos una vez** (S6, S7) — nunca menos, a veces mas, por reintentos ante fallos de red. `dropDuplicates` acotado por watermark es la pieza que convierte ese "al menos una vez" en un resultado que se comporta como "exactamente una vez", **sin** que el productor tenga que garantizar nada.


## 3.11 Intervalo de disparo: el costo de decidir cada cuanto procesar

**Producto del paso:** el mismo stream, con dos ritmos de disparo distintos, y la diferencia real en cuantos micro-lotes se alcanzan a correr.


In [13]:
def contar_microlotes(intervalo, segundos_totales):
    consulta = (
        eventos.writeStream
        .outputMode("append")
        .format("memory")
        .queryName(f"conteo_{intervalo.replace(' ', '_')}")
        .trigger(processingTime=intervalo)
        .start()
    )
    consulta.awaitTermination(segundos_totales)
    consulta.stop()
    return consulta.lastProgress["batchId"] if consulta.lastProgress else None

ultimo_lote_rapido = contar_microlotes("1 second", 15)
ultimo_lote_lento = contar_microlotes("6 seconds", 15)

print("Con disparo cada 1 segundo, ultimo batchId en 15s:", ultimo_lote_rapido)
print("Con disparo cada 6 segundos, ultimo batchId en 15s:", ultimo_lote_lento)


Con disparo cada 1 segundo, ultimo batchId en 15s: 5
Con disparo cada 6 segundos, ultimo batchId en 15s: 2


`format("memory")` guarda la salida en una tabla temporal en vez de imprimirla — util para inspeccionar resultados con SQL, y aca para medir sin llenar la pantalla de filas. `consulta.lastProgress["batchId"]` es el numero del ultimo micro-lote que corrio: con el disparo de 1 segundo, en 15 segundos de reloj corren muchos mas micro-lotes que con el de 6 segundos.

La diferencia es el compromiso entre **latencia** y **throughput** (rendimiento) que pide el silabo: un disparo mas frecuente reduce la latencia (un evento espera menos, en promedio, antes de procesarse) pero cada micro-lote es mas chico y el overhead de coordinarlos se paga mas veces por segundo; un disparo menos frecuente junta mas eventos por lote (mejor throughput por lote) a cambio de que cada evento individual espere mas antes de aparecer en el resultado. Ninguno es "el correcto": depende de si el sistema necesita reaccionar rapido (alertas) o procesar volumen (reportes).


## 3.12 Escribir la salida a Parquet, con checkpoint

**Producto del paso:** los eventos del stream aterrizando en disco, en Parquet, listos para que S9-S10 los lean como datos ya guardados.


In [14]:
ARTIFACTS = "/opt/s08-streaming-estructurado/artifacts/atmos_parquet"
CHECKPOINT_PARQUET = "/opt/s08-streaming-estructurado/artifacts/chk_parquet"

consulta = (
    eventos.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(20)
consulta.stop()

guardado = spark.read.parquet(ARTIFACTS)
print("filas guardadas en Parquet:", guardado.count())
guardado.select("sensorId", "temperatura", "ts").show(5, truncate=False)


filas guardadas en Parquet: 18
+-----------------+-----------+-----------------------+
|sensorId         |temperatura|ts                     |
+-----------------+-----------+-----------------------+
|esp32-patio      |21.1       |2026-10-03 14:04:36.572|
|esp32-invernadero|38.7       |2026-10-03 14:04:36.574|
|esp32-patio      |21.1       |2026-10-03 14:04:39.577|
|esp32-invernadero|38.6       |2026-10-03 14:04:39.579|
|esp32-patio      |20.7       |2026-10-03 14:04:30.56 |
+-----------------+-----------+-----------------------+
only showing top 5 rows


Un *sink* de archivo (Parquet, aca) solo acepta `outputMode("append")` — no puede "reescribir" una fila ya guardada en disco, a diferencia de `console` con `update`. Por eso esta celda escribe los eventos **sin agregar** (3.3), no las ventanas de 3.6-3.7: una agregacion en modo `update` no se puede volcar directo a Parquet.

**Cuidado con `startingOffsets: "latest"` y una corrida corta:** si el productor tarda en emitir su primer evento despues de que la consulta arranca, el primer micro-lote puede quedar vacio, y si la consulta se detiene antes de que llegue el segundo micro-lote, el resultado es **cero filas guardadas** — no por un error, sino porque no hubo datos nuevos en la ventana de tiempo que la consulta estuvo viva. Si esta celda muestra 0 filas, corre de nuevo dejando la consulta activa mas tiempo, o confirma que el productor de 3.1 sigue corriendo.

Vuelve a ejecutar la celda de lectura (`spark.read.parquet(ARTIFACTS)`) despues de correr esta celda una segunda vez: el conteo debe **crecer**, no reiniciarse — la carpeta de Parquet acumula, no sobreescribe.


## 3.13 Documentar hallazgos y responder preguntas de reflexion

**Producto del paso:** una celda markdown que dice, en tus propias palabras, que efecto tuvo cada parametro — sin volver a leer el notebook para recordarlo.

Responde, con los resultados reales que obtuviste arriba:

1. En 3.5, ¿que le pasaba a la tabla impresa en cada micro-lote, y por que dejo de pasar en 3.6?
2. En 3.9, ¿el evento tardio aparecio o no en la salida? ¿Cuantos segundos de diferencia habia entre su `timestamp` y el watermark del momento en que se publico?
3. En 3.10, ¿cuantas veces se publico `esp32-duplicado`, y cuantas veces aparecio en la salida deduplicada?
4. En 3.11, ¿que `batchId` alcanzo cada disparo (1 segundo vs. 6 segundos) en los mismos 15 segundos de reloj? ¿Cual eligirias para un panel de alertas en vivo, y cual para un reporte nocturno?


**Hallazgos de mi corrida (03/10/2026, datos en vivo del productor de S7):**

1. **Sin watermark (3.5)** la tabla en modo `complete` reimprime todas las ventanas en cada micro-lote y crece sin límite; **con watermark de 10 s (3.6)** en modo `update` solo aparecen las ventanas que cambiaron.
2. **Checkpoint (3.8):** contiene `commits`, `metadata`, `offsets`, `sources` y `state`; el último archivo de `offsets/` guardó `{"atmos-eventos":{"0":776,"1":0,"2":2558}}` y el `batchWatermarkMs` del lote.
3. **Dato tardío (3.9):** `esp32-tardio` se publicó con 45 s de antigüedad y no apareció en ninguno de los 7 micro-lotes siguientes: el watermark de 5 s lo descartó en silencio. **Duplicado (3.10):** `esp32-duplicado` se publicó dos veces y apareció una sola vez.
4. **Intervalo de disparo (3.11):** en 15 s, el disparo de 1 s llegó a `batchId` 5 y el de 6 s a `batchId` 2 — más lotes y menor latencia frente a menos *overhead* por lote. **Parquet (3.12):** 18 filas guardadas, con checkpoint.
